## FINAL PROJECT: EXAMINATION OF BACTERIAL GENOME

## PCB - 03701
## NAME: MRUNMAYEE WANKHEDE
## ANDREW ID: mwankhed 

In [1]:
#install packages and libraries
import csv
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.Blast import NCBIWWW, NCBIXML

### Part A

In [2]:
#define a function to find the ORFs (minimum 50 codons)
def find_orfs(sequence, min_codons=50, strand="forward"):
    #define the start codon
    start_codon = "ATG"
    #define the stop codons
    stop_codons = ["TAA", "TAG", "TGA"]
    #create an empty list for the ORFs
    orfs = []

    #find the length of the sequence
    seq_len = len(sequence)
    
    #range is 3 because 3 reading frames, each codon = 3 nucleotides
    for frame in range(3):
        #set the starting position as the current frame
        #loop through the FASTA file as long as the remaining sequence is long enough for an ORF
        i = frame
        while i <= seq_len - 3:

            #extract a codon (3 nucleotide sequence) and check if it is a start codon ("ATG"):
            codon = sequence[i:i+3]
            if codon == start_codon:

                #start scanning for the next codon and move in steps of 3 nucleotides
                #(i+3) sets the start at the 1st codon after the start codon
                #(sequence_len-2) makes sure that the codon is always a full set of 3 nucleotides
                #3, ensures that it jumps (steps) by 3 each time, keeping the scan in the reading frame
                for j in range(i+3, seq_len-2, 3):
                    #extracts a 3-nucleotide sequnce that starts at the position j
                    codon_j = sequence[j:j+3]
                    if codon_j in stop_codons:
                        #measure the ORF length
                        #i is the where the ORF starts
                        #j is where the ORF stops
                        #(j + 3) moves the end to just after the stop codon
                        #(j+3-i) gives the total no. of nucleotides from the beginning of start codon to end of stop codon
                        orf_len = (j+3-i) // 3
                        #check if ORF long enough
                        #store the start, the end, the length, the frame and the strand info
                        #i = start position
                        #(j+3) = end position (including stop codon)
                        #orf_length = length in codons (not nucleotides)
                        #(frame+1) = specifies which reading frame (1,2,3)
                        #forward specifies the strand direction
                        if orf_len >= min_codons:
                            orfs.append((i, j+3, orf_len, frame+1, strand))
                        break
            i = i + 3

    #return the list of the found ORFs
    return orfs

#read sequence from the downloaded FASTA file
filename = "GCF_000063585.1_ASM6358v1_genomic.fna"
all_orfs_forward = []
all_orfs_reverse = []
all_proteins = []

#use SeqI0.parse to read each sequence from the FASTA file
for record in SeqIO.parse(filename, "fasta"):
    #get the DNA sequence as a string
    seq = str(record.seq)
    #get the reverse complement strand, and turn it into a string too
    rc_seq = str(record.seq.reverse_complement())
    #find ORFs on the forward strand
    orfs_fwd = find_orfs(seq, strand="forward")
    #find ORFs on the reverse strand
    orfs_rev = find_orfs(rc_seq, strand="reverse")

    #combine ORFs from this record with the list of all forward strand ORFs
    all_orfs_forward.extend(orfs_fwd)
    #combine ORFs from this record with the list of all reverse strand ORFs
    all_orfs_reverse.extend(orfs_rev)


#this is just to print the first 5 ORFs only
#we are not printing all the ORFs from the entire bacterial genome as there are thousands of ORFs
#go through each ORF found on the forward strand
#start = position where ORF begins (orf[0])
#end = position after the stop codon (orf[1])
#length(codons) = length of the ORf in codons (3 nucleotides) (orf[2])
#frame = specifies the reading frame (1,2,3) (orf[3])
#strand = indicates which DNA strand the ORF is on ('forward' or 'reverse') (orf[4])
#print the total no. of ORFs on the forward strand
print("First 5 forward strand ORFs:")
for orf in all_orfs_forward[:5]:
    print(f"Start: {orf[0]}, End: {orf[1]}, Length(codons): {orf[2]}, Frame: {orf[3]}, Strand: {orf[4]}")

#print the total no. of ORFs on the reverse strand
print("First 5 reverse strand ORFs:")
for orf in all_orfs_reverse[:5]:
    print(f"Start: {orf[0]}, End: {orf[1]}, Length(codons): {orf[2]}, Frame: {orf[3]}, Strand: {orf[4]}")

#create a file called "all_orfs.csv" for writing
with open("all_orfs.csv", "w", newline="") as csvfile:
    #create a CSV writer object which will let us write rows to the file in CSV format
    writer = csv.writer(csvfile)
    #write a header row with column names
    writer.writerow(["Start", "End", "Length_codons", "Frame", "Strand"])
    #loop through the combined list of forward and reverse strand ORFs
    for orf in all_orfs_forward + all_orfs_reverse:
        #write each ORF as a row in the CSV file
        writer.writerow(list(orf))   

First 5 forward strand ORFs:
Start: 0, End: 1347, Length(codons): 449, Frame: 1, Strand: forward
Start: 318, End: 1347, Length(codons): 343, Frame: 1, Strand: forward
Start: 327, End: 1347, Length(codons): 340, Frame: 1, Strand: forward
Start: 480, End: 1347, Length(codons): 289, Frame: 1, Strand: forward
Start: 843, End: 1347, Length(codons): 168, Frame: 1, Strand: forward
First 5 reverse strand ORFs:
Start: 513, End: 690, Length(codons): 59, Frame: 1, Strand: reverse
Start: 744, End: 1080, Length(codons): 112, Frame: 1, Strand: reverse
Start: 1485, End: 1965, Length(codons): 160, Frame: 1, Strand: reverse
Start: 1563, End: 1965, Length(codons): 134, Frame: 1, Strand: reverse
Start: 1773, End: 1965, Length(codons): 64, Frame: 1, Strand: reverse


### Part B

In [3]:
from Bio.Seq import Seq

#create an empty list to store the proteins
all_proteins = []

#translate the forward strand ORFs
#extract ORF information
for start, end, length, frame, strand in orfs_fwd:
    #trim ORF sequence so its length is a multiple of 3 for codon alignment
    dna = seq[start:end]
    dna = dna[:len(dna) - len(dna) % 3]
    #translate DNA sequence to protein, to_stop=True ensures that translation stops at the first stop codon
    protein = str(Seq(dna).translate(to_stop=True))
    #store the translated protein and data as a dictionary
    all_proteins.append({
        "start": start,
        "end": end,
        "length": length,
        "frame": frame,
        "strand": strand,
        "protein_seq": protein
    })

#do the same for the reverse strand
#extract ORF information
for start, end, length, frame, strand in orfs_rev:
    #trim ORF sequence so its length is a multiple of 3 for codon alignment
    dna = rc_seq[start:end]
    dna = dna[:len(dna) - len(dna) % 3] 
    #translate DNA sequence to protein, stop=True ensures that  translation stops at the first stop codon
    protein = str(Seq(dna).translate(to_stop=True)) 
    #store the translated protein and data as a dictionary
    all_proteins.append({
        "start": start,
        "end": end,
        "length": length,
        "frame": frame,
        "strand": strand,
        "protein_seq": protein
    })

print("Total proteins translated:", len(all_proteins))
#printing one protein as an example
print("Example protein:", all_proteins[0])


#create a file called "all_proteins.csv" for writing
with open("all_proteins.csv", "w", newline="") as csvfile:
    #create a CSV writer object
    writer = csv.writer(csvfile)
    #write a header row with column names
    writer.writerow(["Start", "End", "Length", "Frame", "Strand", "Protein_seq"])
    #loop through all protein dictionaries
    for protein_dict in all_proteins:
        #write each protein's data to one row in the CSV
        writer.writerow([
            protein_dict["start"],
            protein_dict["end"],
            protein_dict["length"],
            protein_dict["frame"],
            protein_dict["strand"],
            protein_dict["protein_seq"]
        ])

Total proteins translated: 96
Example protein: {'start': 4776, 'end': 5343, 'length': 189, 'frame': 1, 'strand': 'forward', 'protein_seq': 'MYVRGDKMELVQPIRDKEVINKFKNELLKKGYRNYMLFVIGINTGLRISDILKLKVEDVKDKTHIVIREQKTSKRTRVLINSMLREDIEKYIEGMDNTEYLFKSRNGKNKPITRVQAYRILSEVGKHLGMSEVGTHTLRKTFGYWHYKQYKDVAILQDIFNHSAPSITLKYIGINDDIKDNTIENFYL'}


### Part C

In [4]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

#get the amino acid (AA) sequence
for protein in all_proteins:
    aa_seq = protein["protein_seq"]
    if aa_seq: #calculates only if AA sequence isn't empty
        #calculate the molecular weight using the imported module ProteinAnalysis
        analysis = ProteinAnalysis(aa_seq)
        #calculate molecular weight and convert Da to kDa
        mw = analysis.molecular_weight() / 1000 
    else:
        mw = 0
    #stores the data in dictionary
    protein["molecular_weight_kDa"] = mw

#print the molecular weight of the first 5 proteins
#range(5) makes the loop run for the first 5 proteins
for i in range(5):
    p = all_proteins[i]
    #{i+1} makes sure protein starts from 1 not 0
    print(f"Protein {i+1}: {p['molecular_weight_kDa']} kDa")

Protein 1: 22.1935092 kDa
Protein 2: 21.343512299999997 kDa
Protein 3: 17.894480299999998 kDa
Protein 4: 12.4920737 kDa
Protein 5: 11.014393 kDa


### Part D

In [5]:
#select 5 proteins with sequences shorter than 1000 AA
selected = [p for p in all_proteins if len(p["protein_seq"]) < 1000][:5]

#loop over the selected proteins
#enumerate used to count from 1
for i, protein in enumerate(selected, start=1):
    #print blank line for formatting
    print()
    #print which protein is being processed
    print(f"BLASTing Protein {i}") 
    #print blank line for formatting
    print()
    #run BLASTP against the NCBI nr database for this protein
    result_handle = NCBIWWW.qblast("blastp", "nr", protein["protein_seq"])
    #save BLAST result to unique output file
    file_name = f"blast_result_{i}.xml"
    #save the BLAST result XML to a file
    with open(file_name, "w") as f:
        f.write(result_handle.read())
    print(f"Saved to {file_name}")

    #open and parse the BLAST result file
    with open(file_name) as f:
        blast_record = NCBIXML.read(f)

    #print the number of BLAST hits found
    print("Hits found:", len(blast_record.alignments))
    if blast_record.alignments:
        #for the top 3 hits, print detailed info
        for j, alignment in enumerate(blast_record.alignments[:3], start=1): 
            #take the highest-scoring pair for this alignment
            hsp = alignment.hsps[0]
            print(f"\nHit {j}:")
            print("Title:", alignment.title) #description of the BLAST subject
            print("E-value:", hsp.expect) #E-value for this HSP
            print("Score:", hsp.score) #alignment score for this HSP
            print("Query:", hsp.query[:75] + "...") #first 75 letters of query alignment
            print("Match:", hsp.match[:75] + "...") #first 75 match letters
            print("Subject:", hsp.sbjct[:75] + "...") #first 75 subject alignment letters
    else:
        print("No hits found.") #in case no hits are found


BLASTing Protein 1

Saved to blast_result_1.xml
Hits found: 50

Hit 1:
Title: gb|QGT45393.1| Tyrosine recombinase XerH [Clostridium botulinum]
E-value: 5.53314e-133
Score: 978.0
Query: MYVRGDKMELVQPIRDKEVINKFKNELLKKGYRNYMLFVIGINTGLRISDILKLKVEDVKDKTHIVIREQKTSKR...
Match: MYVRGDKMELVQPIRDKEVINKFKNELLKKGYRNYMLFVIGINTGLRISDILKLKVEDVKDKTHIVIREQKTSKR...
Subject: MYVRGDKMELVQPIRDKEVINKFKNELLKKGYRNYMLFVIGINTGLRISDILKLKVEDVKDKTHIVIREQKTSKR...

Hit 2:
Title: ref|WP_011949169.1| site-specific integrase [Clostridium botulinum] >gb|NFL70834.1| site-specific integrase [Clostridium botulinum] >gb|NFQ55177.1| site-specific integrase [Clostridium botulinum] >gb|NFT48104.1| site-specific integrase [Clostridium botulinum] >emb|CAL81538.1| site-specific recombinase [Clostridium botulinum A str. ATCC 3502]
E-value: 1.73571e-127
Score: 941.0
Query: MELVQPIRDKEVINKFKNELLKKGYRNYMLFVIGINTGLRISDILKLKVEDVKDKTHIVIREQKTSKRTRVLINS...
Match: MELVQPIRDKEVINKFKNELLKKGYRNYMLFVIGINTGLRISDILKLKVEDVKDKTHIVIREQKTSKRTRVLIN